# Chapter 13: Loading and Preprocessing Data with TensorFlow

Bab ini membahas cara membangun *input pipelines* yang efisien menggunakan Data API dari TensorFlow untuk menangani dataset besar yang tidak muat di dalam memori.

---

## 13.1 Data API
Pusat dari Data API adalah `tf.data.Dataset`, yang merepresentasikan sekuens elemen yang bisa diproses secara efisien.
* **`from_tensor_slices()`**: Mengambil tensor dan membuat dataset yang elemen-elemennya diambil dari dimensi pertama tensor tersebut (misal: memecah matriks menjadi baris-baris data).

---

## 13.2 Chaining Transformations
Metode transformasi pada dataset dapat dihubungkan secara berturut-turut (*method chaining*). Setiap metode akan mengembalikan objek dataset baru.
* **`map()`**: Digunakan untuk menerapkan fungsi kustom (seperti prapemrosesan) pada setiap elemen dalam dataset.
* **Contoh lain**: `batch()`, `filter()`, dan `repeat()`.

---

## 13.3 Shuffling Data
Gradient Descent bekerja paling baik jika data dalam *training set* terdistribusi secara independen dan identik (IID).
* **Buffer Shuffling**: Dataset mengisi *buffer* dengan elemen awal, lalu mengambil elemen secara acak dari *buffer* tersebut. Penting untuk menentukan ukuran *buffer* yang cukup besar agar pengacakan efektif.
* **`interleave()`**: Digunakan untuk membaca data dari banyak file secara paralel (misal: 5 file sekaligus) dan menggabungkan hasilnya. Ini membantu meningkatkan performa I/O.

---

## 13.4 Preprocessing Data
Prapemrosesan data (seperti standarisasi atau normalisasi) biasanya dilakukan melalui fungsi kustom (misal: `def preprocess(line):`) yang kemudian dipanggil di dalam metode `.map()`.

---

## 13.5 Prefetching
Teknik ini sangat penting untuk performa:
* **`prefetch(1)`**: Menciptakan dataset yang selalu menyiapkan satu *batch* lebih awal secara paralel.
* Saat GPU sedang melatih satu *batch*, CPU bekerja secara bersamaan untuk menyiapkan *batch* berikutnya. Hal ini meminimalkan waktu tunggu perangkat keras.

---

## 13.6 Menggunakan Dataset dengan tf.keras
Dataset yang telah dibuat (misalnya melalui fungsi `csv_reader_dataset`) dapat langsung dimasukkan ke dalam metode `model.fit()`, `model.evaluate()`, dan `model.predict()` pada Keras. Hal ini membuat integrasi *input pipeline* menjadi sangat mulus.

---

## 13.7 TFRecord Format
TFRecord adalah format biner pilihan TensorFlow untuk menyimpan data dalam jumlah besar dan membacanya secara efisien.
* Format ini hanya mengandung sekuens catatan biner (*binary records*) dengan berbagai ukuran.
* Sangat optimal untuk menangani *throughput* data yang tinggi.

---

## 13.8 Protocol Buffers
Secara internal, TFRecord biasanya berisi *serialized protocol buffers* (protobufs).
* **`tf.train.Example`**: Protobuf standar yang merepresentasikan satu baris data dalam dataset.
* Mengandung **Features** yang terdiri dari:
  * `BytesList`: Untuk string atau data biner.
  * `FloatList`: Untuk angka desimal.
  * `Int64List`: Untuk angka bulat.

---

## 13.9 Penggunaan Lainnya (Loading & Encoding)
* **Parsing**: Menggunakan `tf.io.parse_single_example` untuk mengubah data biner kembali ke bentuk tensor.
* **SequenceExample**: Digunakan jika data bersifat sekuensial (seperti teks atau deret waktu) yang memiliki konteks.
* **Keras Preprocessing Layers**: Standarisasi dapat diimplementasikan langsung sebagai bagian dari model (misalnya menggunakan lapisan `Normalization` atau `Lambda`).
* **One-Hot Encoding**: Untuk mengubah kategori string menjadi representasi angka biner.
* **Embeddings**: Vektor angka yang mewakili kategori. Dibandingkan One-Hot, *embeddings* lebih efisien untuk kategori berjumlah besar dan mendukung *representation learning*.

---

## 13.10 TF Transform (TFT)
Jika prapemrosesan membutuhkan biaya komputasi tinggi, kita bisa menggunakan **TF Transform**.
* Data diproses sekali saja (misal: menghitung *mean* dan *standard deviation* dari seluruh dataset) sebelum pelatihan dimulai.
* Hasil transformasi ini kemudian diekspor sebagai bagian dari grafik TensorFlow agar konsisten saat fase *deployment*.